In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sb
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import r2_score, mean_absolute_error
ohe = OneHotEncoder()


In [3]:
df = pd.read_csv("datasets/RTA Dataset.csv")
df.head()

,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Vehicle_movement,Casualty_class,Sex_of_casualty,Age_band_of_casualty,Casualty_severity,Work_of_casuality,Fitness_of_casuality,Pedestrian_movement,Cause_of_accident,Accident_severity
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Moving Backward,Slight Injury
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,NaN,...,Going straight,Driver or rider,Male,31-50,3,Driver,NaN,Not a Pedestrian,Changing lane to the left,Serious Injury
3,1:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,NaN,...,Going straight,Pedestrian,Female,18-30,3,Driver,Normal,Not a Pedestrian,Changing lane to the right,Slight Injury
4,1:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,5-10yrs,...,Going straight,na,na,na,na,NaN,NaN,Not a Pedestrian,Overtaking,Slight Injury


In [4]:
df['Educational_level'].unique()

array(['Above high school', 'Junior high school', nan,
       'Elementary school', 'High school', 'Unknown', 'Illiterate',
       'Writing & reading'], dtype=object)

In [48]:
df.drop(columns=['x','y','z','table','depth'], inplace=True)

In [49]:
df.head()

,carat,cut,color,clarity,price
0,0.23,Ideal,E,SI2,326
1,0.21,Premium,E,SI1,326
2,0.23,Good,E,VS1,327
3,0.29,Premium,I,VS2,334
4,0.31,Good,J,SI2,335


In [50]:
df.isnull().sum()

carat      0
cut        0
color      0
clarity    0
price      0
dtype: int64

In [51]:
df.shape

(53940, 5)

In [52]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   price    53940 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 2.1+ MB


In [53]:
# Define custom orderings
cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Create encoder with specified categories
encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])

# Fit and transform
df_encoded = df.copy()
df_encoded[['cut', 'color', 'clarity']] = encoder.fit_transform(df[['cut', 'color', 'clarity']])

# Show encoded data
print(df_encoded)

       carat  cut  color  clarity  price
0       0.23  4.0    5.0      1.0    326
1       0.21  3.0    5.0      2.0    326
2       0.23  1.0    5.0      4.0    327
3       0.29  3.0    1.0      3.0    334
4       0.31  1.0    0.0      1.0    335
...      ...  ...    ...      ...    ...
53935   0.72  4.0    6.0      2.0   2757
53936   0.72  1.0    6.0      2.0   2757
53937   0.70  2.0    6.0      2.0   2757
53938   0.86  3.0    2.0      1.0   2757
53939   0.75  4.0    6.0      1.0   2757

[53940 rows x 5 columns]


In [54]:
x = df_encoded.drop(columns=['price'])
y = df_encoded['price']

In [55]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size = 0.2, random_state=42)
print(x_train.shape)
print(x_test.shape)

(43152, 4)
(10788, 4)


In [56]:
model = LinearRegression()
model.fit(x_train,y_train)

LinearRegression()

In [57]:
y_pred = model.predict(x_test)

In [58]:
r2 = r2_score(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
print(f"R2 score is {r2}")
print(f"Mean absolute error is {mae}")

R2 score is 0.9030563214406131
Mean absolute error is 855.7062135327616


In [59]:
def adjusted_r2(r2, n, k):
    return 1 - (1 - r2) * ((n - 1) / (n - k - 1))

adj_r2 = adjusted_r2(r2, n=len(y_train), k=x.shape[1])
print("Adjusted R-squared:", adj_r2)

Adjusted R-squared: 0.9030473341480032


In [60]:
import statsmodels.api as sm

X_with_const = sm.add_constant(x)
model = sm.OLS(y, X_with_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.904
Model:                            OLS   Adj. R-squared:                  0.904
Method:                 Least Squares   F-statistic:                 1.272e+05
Date:                Wed, 30 Jul 2025   Prob (F-statistic):               0.00
Time:                        16:25:28   Log-Likelihood:            -4.6053e+05
No. Observations:               53940   AIC:                         9.211e+05
Df Residuals:                   53935   BIC:                         9.211e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const      -6218.3384     27.054   -229.851      0.0

In [61]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Assuming X is your feature dataframe
vif_data = pd.DataFrame()
vif_data["feature"] = x.columns
vif_data["VIF"] = [variance_inflation_factor(x.values, i) for i in range(x.shape[1])]

print(vif_data)


   feature       VIF
0    carat  2.587821
1      cut  6.176619
2    color  3.636132
3  clarity  3.897282
